# 🛰️ AFETSONAR — Notebook 4: Koordinat ve Mozaikleme

**Calamitas AI · Teknofest 2025 · Plan v2.0 · Faz 4**

---

## 📌 Bu Notebook Ne Yapıyor?

Bu notebook, modelimizin **çıktısını gerçek dünyaya bağlamak** için gerekli alt yapıyı kuruyor.

### Sorun

Modelimiz bir görüntüyü segmentasyona uğratıyor — çıktı pixel düzeyinde:

```
"Pixel (245, 387) destroyed sınıfında"
```

Ama Faz 5'te ekip atayacak, Faz 6'da rota planlayacağız. Bunlar için pixel değil **gerçek dünya koordinatları** lazım:

```
"Lat 41.0123, Lon 28.9876 — destroyed bina"
```

### Çözüm

İki kaynaktan koordinat çıkarabiliriz:

1. **Drone JPEG'leri** → EXIF metadata GPS koordinatı içerir (DJI, Parrot, Autel hepsi yazar)
2. **Uydu GeoTIFF'leri** → CRS (Coordinate Reference System) + affine transform pixel↔geo dönüşümünü tanımlar

### Bu notebook ne yapacak?

1. `src/geo_utils.py` modülünü Drive'a yaz
2. **EXIF GPS okuma** — örnek bir görselden GPS çıkar
3. **GeoTIFF metadata** — CRS, bounds, transform okuma
4. **Pixel↔Geo dönüşümü** — affine transform matematiği
5. **WGS84↔UTM** — metre bazında mesafe hesabı için (Folium WGS84, A* ise metre ister)
6. **Drone footprint hesabı** — GSD (Ground Sample Distance) formülü
7. **Image index** — bir klasördeki tüm görüntüleri tara, GPS'i olanları listele
8. **Sentetik test verisi** — kullanıcının elinde drone görüntüsü olmayabilir, biz dummy üretip test ederiz

### Bu notebook bitince elinde ne olacak?

`src/geo_utils.py` modülü — Notebook 5, 6, 7 hep bunu kullanacak. Tek bir yerden geo-spatial işlemler yönetilecek.

### ⏱️ Süre

Çok hızlı, **5-10 dakika** sürer. Eğitim bitmeden bitirebilirsin.

---

## 1️⃣ Drive Bağlantısı + Yolları Tanımlama

### Bu hücre ne yapıyor?

Notebook 2'deki ile aynı: Drive mount, yolları tanımla, sanity check.

### Hata olursa

- "Bazı klasör/dosyalar eksik!" → Notebook 1'i çalıştırdın mı?

In [1]:
# === Drive Mount ===
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# === Yollar (Notebook 1 ile birebir aynı) ===
import os

PROJECT_ROOT = "/content/drive/MyDrive/AFETSONAR"
DATA_RAW     = os.path.join(PROJECT_ROOT, "data/raw/xview2")
DATA_MASKS   = os.path.join(PROJECT_ROOT, "data/processed/masks")
DATA_SPLITS  = os.path.join(PROJECT_ROOT, "data/splits")
SRC_DIR      = os.path.join(PROJECT_ROOT, "src")
CKPT_TEACHER = os.path.join(PROJECT_ROOT, "checkpoints/teacher")
CKPT_STUDENT = os.path.join(PROJECT_ROOT, "checkpoints/student")
OUTPUTS_VIZ  = os.path.join(PROJECT_ROOT, "outputs/visualizations")

required_dirs = [PROJECT_ROOT, DATA_SPLITS, SRC_DIR]
print("✅ Drive bağlandı")
print()
print("📁 Yollar tanımlandı:")
for path in required_dirs:
    marker = "✅" if os.path.exists(path) else "❌"
    print(f"  {marker} {path}")

if not all(os.path.exists(p) for p in required_dirs):
    raise FileNotFoundError("Notebook 1'i baştan sona çalıştır")

# Test verileri için klasör oluştur
TEST_GEO_DIR = os.path.join(PROJECT_ROOT, "data/test_geo")
os.makedirs(TEST_GEO_DIR, exist_ok=True)
os.makedirs(OUTPUTS_VIZ, exist_ok=True)

print("\n✅ Hazır")

Mounted at /content/drive
✅ Drive bağlandı

📁 Yollar tanımlandı:
  ✅ /content/drive/MyDrive/AFETSONAR
  ✅ /content/drive/MyDrive/AFETSONAR/data/splits
  ✅ /content/drive/MyDrive/AFETSONAR/src

✅ Hazır


## 2️⃣ Coğrafi Kütüphaneleri Hazırla

### Bu hücre ne yapıyor?

Notebook 1'de zaten kurmuş olmamız gereken coğrafi kütüphaneleri tekrar kontrol ediyor:

- `exifread` — drone JPEG'lerinden GPS okuma
- `rasterio` — GeoTIFF okuma
- `pyproj` — koordinat sistem dönüşümleri (WGS84 ↔ UTM)
- `shapely` — geometri işlemleri

Eğer eksikse kurar. `pyproj` Notebook 1'de listelenmemişti — onu yeni ekliyoruz.

In [2]:
print("📦 Coğrafi kütüphane kontrolü...")
print("=" * 50)

def check_or_install(name, import_name=None):
    import_name = import_name or name
    try:
        mod = __import__(import_name)
        ver = getattr(mod, '__version__', 'unknown')
        print(f"  ✅ {name:15s} {ver}")
        return True
    except ImportError:
        print(f"  ⏳ {name:15s} kuruluyor...")
        return False

need_install = []
for pkg, imp in [
    ("exifread", "exifread"),
    ("rasterio", "rasterio"),
    ("pyproj", "pyproj"),
    ("shapely", "shapely"),
]:
    if not check_or_install(pkg, imp):
        need_install.append(pkg)

if need_install:
    print(f"\n📥 Eksik paketler kuruluyor: {need_install}")
    !pip install -q {' '.join(need_install)}
    print("✅ Kurulum tamam")

# Final import testi
print("\n🧪 Import testi...")
import exifread
import rasterio
from pyproj import Transformer
from shapely.geometry import Polygon, Point
print("  ✅ exifread")
print("  ✅ rasterio")
print("  ✅ pyproj.Transformer")
print("  ✅ shapely.geometry")

print("\n✅ Tüm coğrafi kütüphaneler hazır")

📦 Coğrafi kütüphane kontrolü...
  ⏳ exifread        kuruluyor...
  ✅ rasterio        1.5.0
  ✅ pyproj          3.7.2
  ✅ shapely         2.1.2

📥 Eksik paketler kuruluyor: ['exifread']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 1.7 MB/s eta 0:00:00
✅ Kurulum tamam

🧪 Import testi...
  ✅ exifread
  ✅ rasterio
  ✅ pyproj.Transformer
  ✅ shapely.geometry

✅ Tüm coğrafi kütüphaneler hazır


## 3️⃣ `src/geo_utils.py` Modülünü Yazma

### Bu hücre ne yapıyor?

`src/` klasörüne `geo_utils.py` modülünü ekliyor. İçinde şu fonksiyonlar var:

| Fonksiyon | Ne yapar? |
|---|---|
| `read_exif_gps(image_path)` | Drone JPEG'inden GPS koordinatı okur |
| `read_exif_camera_info(image_path)` | Kamera modeli, focal length, vs |
| `read_geotiff_metadata(tiff_path)` | GeoTIFF CRS + bounds + transform |
| `read_geotiff_array(tiff_path)` | GeoTIFF'i numpy array olarak oku |
| `pixel_to_geo(px, py, transform)` | Pixel → coğrafi koordinat |
| `geo_to_pixel(x, y, transform)` | Coğrafi → pixel (ters) |
| `pixel_polygon_to_geo(poly, transform)` | Bir poligonu geo'ya çevir (bina sınırları için) |
| `wgs84_to_utm(lat, lon)` | WGS84 → UTM (metre cinsinden) |
| `utm_to_wgs84(e, n, zone)` | UTM → WGS84 (ters) |
| `haversine_distance(lat1, lon1, lat2, lon2)` | İki nokta arası metre |
| `estimate_drone_footprint(...)` | GSD hesabı (ground sample distance) |
| `build_image_index(image_dir)` | Bir klasördeki tüm görüntüleri tara, GPS'i olanları listele |

Bu modülü Notebook 5 (priority), 6 (routing), 7 (folium) hepsi kullanacak.

In [3]:
import os
import base64

GEO_UTILS_PY = base64.b64decode("IiIiCkFGRVRTT05BUiBHZW9ncmFwaGljIFV0aWxpdGllcy4KCkZ1bmN0aW9ucyBmb3I6Ci0gUmVhZGluZyBkcm9uZSBHUFMgbWV0YWRhdGEgZnJvbSBFWElGCi0gUmVhZGluZyBHZW9USUZGIGNvb3JkaW5hdGUgc3lzdGVtcyBhbmQgdHJhbnNmb3JtcwotIFBpeGVsIDwtPiBnZW9ncmFwaGljIGNvb3JkaW5hdGUgY29udmVyc2lvbgotIEltYWdlIG1vc2FpY2tpbmcgZnJvbSBtdWx0aXBsZSBzb3VyY2VzCi0gV0dTODQgPC0+IFVUTSBjb252ZXJzaW9uIGZvciBtZXRyaWMgZGlzdGFuY2UgY2FsY3VsYXRpb25zCiIiIgoKaW1wb3J0IG9zCmltcG9ydCBtYXRoCmltcG9ydCB3YXJuaW5ncwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsLCBUdXBsZSwgTGlzdCwgRGljdAoKaW1wb3J0IG51bXB5IGFzIG5wCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyBFWElGIC8gRFJPTkUgR1BTCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgX2NvbnZlcnRfdG9fZGVncmVlcyh2YWx1ZSk6CiAgICAiIiJDb252ZXJ0IEVYSUYgR1BTIHJhdGlvbmFsIGNvb3JkaW5hdGVzIHRvIGRlY2ltYWwgZGVncmVlcy4iIiIKICAgIHRyeToKICAgICAgICBkID0gZmxvYXQodmFsdWVbMF0pCiAgICAgICAgbSA9IGZsb2F0KHZhbHVlWzFdKQogICAgICAgIHMgPSBmbG9hdCh2YWx1ZVsyXSkKICAgICAgICByZXR1cm4gZCArIChtIC8gNjAuMCkgKyAocyAvIDM2MDAuMCkKICAgIGV4Y2VwdCAoVHlwZUVycm9yLCBJbmRleEVycm9yLCBWYWx1ZUVycm9yKToKICAgICAgICByZXR1cm4gTm9uZQoKCmRlZiByZWFkX2V4aWZfZ3BzKGltYWdlX3BhdGg6IHN0cikgLT4gT3B0aW9uYWxbRGljdFtzdHIsIGZsb2F0XV06CiAgICAiIiIKICAgIFJlYWQgR1BTIGNvb3JkaW5hdGVzIGZyb20gYSBkcm9uZSBpbWFnZSdzIEVYSUYgbWV0YWRhdGEuCgogICAgQXJnczoKICAgICAgICBpbWFnZV9wYXRoOiBQYXRoIHRvIEpQRUcvVElGRiBpbWFnZQoKICAgIFJldHVybnM6CiAgICAgICAgZGljdCB3aXRoIGtleXM6IGxhdGl0dWRlLCBsb25naXR1ZGUsIGFsdGl0dWRlIChvciBOb25lIGlmIG5vdCBhdmFpbGFibGUpCiAgICAgICAgUmV0dXJucyBOb25lIGlmIG5vIEdQUyBkYXRhIGZvdW5kLgoKICAgIEV4YW1wbGU6CiAgICAgICAgZ3BzID0gcmVhZF9leGlmX2dwcygiZHJvbmVfcGhvdG8uanBnIikKICAgICAgICBpZiBncHM6CiAgICAgICAgICAgIHByaW50KGYiTGF0OiB7Z3BzWydsYXRpdHVkZSddfSwgTG9uOiB7Z3BzWydsb25naXR1ZGUnXX0iKQogICAgIiIiCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IGV4aWZyZWFkCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgcmFpc2UgSW1wb3J0RXJyb3IoImV4aWZyZWFkIHBha2V0aSBnZXJla2xpOiBwaXAgaW5zdGFsbCBleGlmcmVhZCIpCgogICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKGltYWdlX3BhdGgpOgogICAgICAgIHJldHVybiBOb25lCgogICAgdHJ5OgogICAgICAgIHdpdGggb3BlbihpbWFnZV9wYXRoLCAicmIiKSBhcyBmOgogICAgICAgICAgICB0YWdzID0gZXhpZnJlYWQucHJvY2Vzc19maWxlKGYsIGRldGFpbHM9RmFsc2UsIHN0b3BfdGFnPSJHUFMgR1BTQWx0aXR1ZGUiKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gTm9uZQoKICAgIGlmIG5vdCB0YWdzOgogICAgICAgIHJldHVybiBOb25lCgogICAgIyBMYXRpdHVkZQogICAgbGF0X3RhZyA9IHRhZ3MuZ2V0KCJHUFMgR1BTTGF0aXR1ZGUiKQogICAgbGF0X3JlZiA9IHRhZ3MuZ2V0KCJHUFMgR1BTTGF0aXR1ZGVSZWYiKQogICAgaWYgbGF0X3RhZyBpcyBOb25lIG9yIGxhdF9yZWYgaXMgTm9uZToKICAgICAgICByZXR1cm4gTm9uZQoKICAgIGxhdGl0dWRlID0gX2NvbnZlcnRfdG9fZGVncmVlcyhsYXRfdGFnLnZhbHVlcykKICAgIGlmIGxhdGl0dWRlIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIGlmIHN0cihsYXRfcmVmKSA9PSAiUyI6CiAgICAgICAgbGF0aXR1ZGUgPSAtbGF0aXR1ZGUKCiAgICAjIExvbmdpdHVkZQogICAgbG9uX3RhZyA9IHRhZ3MuZ2V0KCJHUFMgR1BTTG9uZ2l0dWRlIikKICAgIGxvbl9yZWYgPSB0YWdzLmdldCgiR1BTIEdQU0xvbmdpdHVkZVJlZiIpCiAgICBpZiBsb25fdGFnIGlzIE5vbmUgb3IgbG9uX3JlZiBpcyBOb25lOgogICAgICAgIHJldHVybiBOb25lCgogICAgbG9uZ2l0dWRlID0gX2NvbnZlcnRfdG9fZGVncmVlcyhsb25fdGFnLnZhbHVlcykKICAgIGlmIGxvbmdpdHVkZSBpcyBOb25lOgogICAgICAgIHJldHVybiBOb25lCiAgICBpZiBzdHIobG9uX3JlZikgPT0gIlciOgogICAgICAgIGxvbmdpdHVkZSA9IC1sb25naXR1ZGUKCiAgICAjIEFsdGl0dWRlIChvcHRpb25hbCkKICAgIGFsdGl0dWRlID0gTm9uZQogICAgYWx0X3RhZyA9IHRhZ3MuZ2V0KCJHUFMgR1BTQWx0aXR1ZGUiKQogICAgaWYgYWx0X3RhZyBpcyBub3QgTm9uZToKICAgICAgICB0cnk6CiAgICAgICAgICAgIGFsdF92YWwgPSBhbHRfdGFnLnZhbHVlc1swXQogICAgICAgICAgICBhbHRpdHVkZSA9IGZsb2F0KGFsdF92YWwubnVtKSAvIGZsb2F0KGFsdF92YWwuZGVuKQogICAgICAgICAgICBhbHRfcmVmID0gdGFncy5nZXQoIkdQUyBHUFNBbHRpdHVkZVJlZiIpCiAgICAgICAgICAgIGlmIGFsdF9yZWYgaXMgbm90IE5vbmUgYW5kIGludChzdHIoYWx0X3JlZikpID09IDE6CiAgICAgICAgICAgICAgICBhbHRpdHVkZSA9IC1hbHRpdHVkZQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIGFsdGl0dWRlID0gTm9uZQoKICAgIHJldHVybiB7CiAgICAgICAgImxhdGl0dWRlIjogbGF0aXR1ZGUsCiAgICAgICAgImxvbmdpdHVkZSI6IGxvbmdpdHVkZSwKICAgICAgICAiYWx0aXR1ZGUiOiBhbHRpdHVkZSwKICAgIH0KCgpkZWYgcmVhZF9leGlmX2NhbWVyYV9pbmZvKGltYWdlX3BhdGg6IHN0cikgLT4gRGljdFtzdHIsIGFueV06CiAgICAiIiIKICAgIFJlYWQgY2FtZXJhL2Ryb25lIG1ldGFkYXRhOiBtb2RlbCwgZm9jYWwgbGVuZ3RoLCBpbWFnZSBzaXplLCB0aW1lc3RhbXAuCiAgICBVc2VmdWwgZm9yIHVuZGVyc3RhbmRpbmcgR1NEIChncm91bmQgc2FtcGxlIGRpc3RhbmNlKSBhbmQgaW1hZ2Ugc2NhbGUuCiAgICAiIiIKICAgIHRyeToKICAgICAgICBpbXBvcnQgZXhpZnJlYWQKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICByZXR1cm4ge30KCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoaW1hZ2VfcGF0aCk6CiAgICAgICAgcmV0dXJuIHt9CgogICAgdHJ5OgogICAgICAgIHdpdGggb3BlbihpbWFnZV9wYXRoLCAicmIiKSBhcyBmOgogICAgICAgICAgICB0YWdzID0gZXhpZnJlYWQucHJvY2Vzc19maWxlKGYsIGRldGFpbHM9RmFsc2UpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiB7fQoKICAgIGluZm8gPSB7fQogICAgaWYgIkltYWdlIE1ha2UiIGluIHRhZ3M6CiAgICAgICAgaW5mb1sibWFrZSJdID0gc3RyKHRhZ3NbIkltYWdlIE1ha2UiXSkKICAgIGlmICJJbWFnZSBNb2RlbCIgaW4gdGFnczoKICAgICAgICBpbmZvWyJtb2RlbCJdID0gc3RyKHRhZ3NbIkltYWdlIE1vZGVsIl0pCiAgICBpZiAiRVhJRiBGb2NhbExlbmd0aCIgaW4gdGFnczoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGZsID0gdGFnc1siRVhJRiBGb2NhbExlbmd0aCJdLnZhbHVlc1swXQogICAgICAgICAgICBpbmZvWyJmb2NhbF9sZW5ndGhfbW0iXSA9IGZsb2F0KGZsLm51bSkgLyBmbG9hdChmbC5kZW4pCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcGFzcwogICAgaWYgIkVYSUYgRXhpZkltYWdlV2lkdGgiIGluIHRhZ3M6CiAgICAgICAgaW5mb1siaW1hZ2Vfd2lkdGgiXSA9IGludChzdHIodGFnc1siRVhJRiBFeGlmSW1hZ2VXaWR0aCJdKSkKICAgIGlmICJFWElGIEV4aWZJbWFnZUxlbmd0aCIgaW4gdGFnczoKICAgICAgICBpbmZvWyJpbWFnZV9oZWlnaHQiXSA9IGludChzdHIodGFnc1siRVhJRiBFeGlmSW1hZ2VMZW5ndGgiXSkpCiAgICBpZiAiRVhJRiBEYXRlVGltZU9yaWdpbmFsIiBpbiB0YWdzOgogICAgICAgIGluZm9bImRhdGV0aW1lIl0gPSBzdHIodGFnc1siRVhJRiBEYXRlVGltZU9yaWdpbmFsIl0pCgogICAgcmV0dXJuIGluZm8KCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojIEdFT1RJRkYgLyBSQVNURVIKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiByZWFkX2dlb3RpZmZfbWV0YWRhdGEodGlmZl9wYXRoOiBzdHIpIC0+IERpY3Rbc3RyLCBhbnldOgogICAgIiIiCiAgICBSZWFkIGNvb3JkaW5hdGUgbWV0YWRhdGEgZnJvbSBhIEdlb1RJRkYgZmlsZS4KCiAgICBSZXR1cm5zOgogICAgICAgIGRpY3Qgd2l0aDogY3JzLCBib3VuZHMgKGxlZnQvYm90dG9tL3JpZ2h0L3RvcCksIHRyYW5zZm9ybSwKICAgICAgICAgICAgICAgICAgIHdpZHRoLCBoZWlnaHQsIHBpeGVsX3NpemVfeCwgcGl4ZWxfc2l6ZV95CiAgICAiIiIKICAgIHRyeToKICAgICAgICBpbXBvcnQgcmFzdGVyaW8KICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICByYWlzZSBJbXBvcnRFcnJvcigicmFzdGVyaW8gcGFrZXRpIGdlcmVrbGk6IHBpcCBpbnN0YWxsIHJhc3RlcmlvIikKCiAgICB3aXRoIHJhc3RlcmlvLm9wZW4odGlmZl9wYXRoKSBhcyBzcmM6CiAgICAgICAgYm91bmRzID0gc3JjLmJvdW5kcwogICAgICAgIHRyYW5zZm9ybSA9IHNyYy50cmFuc2Zvcm0KICAgICAgICByZXR1cm4gewogICAgICAgICAgICAiY3JzIjogc3RyKHNyYy5jcnMpIGlmIHNyYy5jcnMgZWxzZSBOb25lLAogICAgICAgICAgICAiYm91bmRzIjogewogICAgICAgICAgICAgICAgImxlZnQiOiBib3VuZHMubGVmdCwKICAgICAgICAgICAgICAgICJib3R0b20iOiBib3VuZHMuYm90dG9tLAogICAgICAgICAgICAgICAgInJpZ2h0IjogYm91bmRzLnJpZ2h0LAogICAgICAgICAgICAgICAgInRvcCI6IGJvdW5kcy50b3AsCiAgICAgICAgICAgIH0sCiAgICAgICAgICAgICJ0cmFuc2Zvcm0iOiBsaXN0KHRyYW5zZm9ybSlbOjZdLAogICAgICAgICAgICAid2lkdGgiOiBzcmMud2lkdGgsCiAgICAgICAgICAgICJoZWlnaHQiOiBzcmMuaGVpZ2h0LAogICAgICAgICAgICAiY291bnQiOiBzcmMuY291bnQsCiAgICAgICAgICAgICJkdHlwZSI6IHN0cihzcmMuZHR5cGVzWzBdKSwKICAgICAgICAgICAgInBpeGVsX3NpemVfeCI6IGFicyh0cmFuc2Zvcm1bMF0pLAogICAgICAgICAgICAicGl4ZWxfc2l6ZV95IjogYWJzKHRyYW5zZm9ybVs0XSksCiAgICAgICAgfQoKCmRlZiByZWFkX2dlb3RpZmZfYXJyYXkodGlmZl9wYXRoOiBzdHIsIGJhbmRzOiBPcHRpb25hbFtMaXN0W2ludF1dID0gTm9uZSkgLT4gVHVwbGVbbnAubmRhcnJheSwgRGljdF06CiAgICAiIiIKICAgIFJlYWQgR2VvVElGRiBhcyBudW1weSBhcnJheSB3aXRoIG1ldGFkYXRhLgoKICAgIEFyZ3M6CiAgICAgICAgdGlmZl9wYXRoOiBwYXRoIHRvIEdlb1RJRkYKICAgICAgICBiYW5kczogbGlzdCBvZiBiYW5kIGluZGljZXMgKDEtaW5kZXhlZCkgdG8gcmVhZDsgTm9uZSByZWFkcyBhbGwKCiAgICBSZXR1cm5zOgogICAgICAgIChhcnJheSwgbWV0YWRhdGEpCiAgICAgICAgYXJyYXkgc2hhcGU6IFtILCBXXSBmb3Igc2luZ2xlIGJhbmQsIFtILCBXLCBDXSBmb3IgbXVsdGktYmFuZAogICAgIiIiCiAgICBpbXBvcnQgcmFzdGVyaW8KCiAgICB3aXRoIHJhc3RlcmlvLm9wZW4odGlmZl9wYXRoKSBhcyBzcmM6CiAgICAgICAgaWYgYmFuZHMgaXMgTm9uZToKICAgICAgICAgICAgYXJyID0gc3JjLnJlYWQoKSAgIyBbQywgSCwgV10KICAgICAgICBlbHNlOgogICAgICAgICAgICBhcnIgPSBzcmMucmVhZChiYW5kcykKCiAgICAgICAgIyBDb252ZXJ0IHRvIFtILCBXXSBvciBbSCwgVywgQ10KICAgICAgICBpZiBhcnIuc2hhcGVbMF0gPT0gMToKICAgICAgICAgICAgYXJyID0gYXJyWzBdCiAgICAgICAgZWxzZToKICAgICAgICAgICAgYXJyID0gbnAudHJhbnNwb3NlKGFyciwgKDEsIDIsIDApKQoKICAgICAgICBtZXRhID0gcmVhZF9nZW90aWZmX21ldGFkYXRhKHRpZmZfcGF0aCkKICAgICAgICByZXR1cm4gYXJyLCBtZXRhCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyBQSVhFTCA8LT4gR0VPIENPT1JESU5BVEUgQ09OVkVSU0lPTgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKZGVmIHBpeGVsX3RvX2dlbygKICAgIHBpeGVsX3g6IGZsb2F0LAogICAgcGl4ZWxfeTogZmxvYXQsCiAgICB0cmFuc2Zvcm06IExpc3RbZmxvYXRdLAopIC0+IFR1cGxlW2Zsb2F0LCBmbG9hdF06CiAgICAiIiIKICAgIENvbnZlcnQgcGl4ZWwgY29vcmRpbmF0ZXMgdG8gZ2VvZ3JhcGhpYyBjb29yZGluYXRlcyB1c2luZyBhZmZpbmUgdHJhbnNmb3JtLgoKICAgIEFmZmluZSB0cmFuc2Zvcm0gZm9ybWF0IChyYXN0ZXJpby9HREFMKToKICAgICAgICBbYSwgYiwgYywgZCwgZSwgZl0gIHdoZXJlCiAgICAgICAgeF9nZW8gPSBhICogcGl4ZWxfeCArIGIgKiBwaXhlbF95ICsgYwogICAgICAgIHlfZ2VvID0gZCAqIHBpeGVsX3ggKyBlICogcGl4ZWxfeSArIGYKCiAgICBBcmdzOgogICAgICAgIHBpeGVsX3g6IGNvbHVtbiBpbmRleCAoMCA9IGxlZnQpCiAgICAgICAgcGl4ZWxfeTogcm93IGluZGV4ICgwID0gdG9wKQogICAgICAgIHRyYW5zZm9ybTogNi1lbGVtZW50IGFmZmluZSB0cmFuc2Zvcm0gbGlzdAoKICAgIFJldHVybnM6CiAgICAgICAgKHhfZ2VvLCB5X2dlbykgaW4gQ1JTIHVuaXRzIChkZWdyZWVzIGZvciBXR1M4NCwgbWV0ZXJzIGZvciBVVE0pCiAgICAiIiIKICAgIGEsIGIsIGMsIGQsIGUsIGYgPSB0cmFuc2Zvcm1bOjZdCiAgICB4X2dlbyA9IGEgKiBwaXhlbF94ICsgYiAqIHBpeGVsX3kgKyBjCiAgICB5X2dlbyA9IGQgKiBwaXhlbF94ICsgZSAqIHBpeGVsX3kgKyBmCiAgICByZXR1cm4geF9nZW8sIHlfZ2VvCgoKZGVmIGdlb190b19waXhlbCgKICAgIHhfZ2VvOiBmbG9hdCwKICAgIHlfZ2VvOiBmbG9hdCwKICAgIHRyYW5zZm9ybTogTGlzdFtmbG9hdF0sCikgLT4gVHVwbGVbZmxvYXQsIGZsb2F0XToKICAgICIiIgogICAgSW52ZXJzZTogZ2VvZ3JhcGhpYyBjb29yZGluYXRlcyAtPiBwaXhlbCBjb29yZGluYXRlcy4KICAgICIiIgogICAgYSwgYiwgYywgZCwgZSwgZiA9IHRyYW5zZm9ybVs6Nl0KICAgICMgU29sdmUgdGhlIDJ4MiBsaW5lYXIgc3lzdGVtCiAgICBkZXQgPSBhICogZSAtIGIgKiBkCiAgICBpZiBhYnMoZGV0KSA8IDFlLTEyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlNpbmd1bGFyIGFmZmluZSB0cmFuc2Zvcm0sIGNhbm5vdCBpbnZlcnQiKQoKICAgIHBpeGVsX3ggPSAoZSAqICh4X2dlbyAtIGMpIC0gYiAqICh5X2dlbyAtIGYpKSAvIGRldAogICAgcGl4ZWxfeSA9ICgtZCAqICh4X2dlbyAtIGMpICsgYSAqICh5X2dlbyAtIGYpKSAvIGRldAogICAgcmV0dXJuIHBpeGVsX3gsIHBpeGVsX3kKCgpkZWYgcGl4ZWxfcG9seWdvbl90b19nZW8oCiAgICBwaXhlbF9wb2x5Z29uOiBMaXN0W1R1cGxlW2Zsb2F0LCBmbG9hdF1dLAogICAgdHJhbnNmb3JtOiBMaXN0W2Zsb2F0XSwKKSAtPiBMaXN0W1R1cGxlW2Zsb2F0LCBmbG9hdF1dOgogICAgIiIiCiAgICBDb252ZXJ0IGEgcGl4ZWwtc3BhY2UgcG9seWdvbiAobGlzdCBvZiBbeCx5XSBwaXhlbCBjb29yZHMpIHRvIGdlb2dyYXBoaWMgY29vcmRzLgoKICAgIFVzZWZ1bCBmb3IgY29udmVydGluZyBidWlsZGluZyBkYW1hZ2UgcG9seWdvbnMgKGZyb20gbW9kZWwgcHJlZGljdGlvbnMgaW4gcGl4ZWwKICAgIHNwYWNlKSB0byBtYXAgY29vcmRpbmF0ZXMgZm9yIEZvbGl1bSB2aXN1YWxpemF0aW9uLgogICAgIiIiCiAgICByZXR1cm4gW3BpeGVsX3RvX2dlbyhweCwgcHksIHRyYW5zZm9ybSkgZm9yIHB4LCBweSBpbiBwaXhlbF9wb2x5Z29uXQoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgV0dTODQgPC0+IFVUTSBDT05WRVJTSU9OCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgZ2V0X3V0bV96b25lKGxvbmdpdHVkZTogZmxvYXQsIGxhdGl0dWRlOiBmbG9hdCkgLT4gVHVwbGVbaW50LCBzdHJdOgogICAgIiIiCiAgICBEZXRlcm1pbmUgVVRNIHpvbmUgKG51bWJlciwgaGVtaXNwaGVyZSkgZm9yIGEgZ2l2ZW4gbGF0L2xvbi4KCiAgICBSZXR1cm5zOgogICAgICAgICh6b25lX251bWJlciwgaGVtaXNwaGVyZSkgd2hlcmUgaGVtaXNwaGVyZSBpcyAnTicgb3IgJ1MnCiAgICAiIiIKICAgIHpvbmVfbnVtYmVyID0gaW50KChsb25naXR1ZGUgKyAxODApIC8gNikgKyAxCiAgICBoZW1pc3BoZXJlID0gIk4iIGlmIGxhdGl0dWRlID49IDAgZWxzZSAiUyIKICAgIHJldHVybiB6b25lX251bWJlciwgaGVtaXNwaGVyZQoKCmRlZiB3Z3M4NF90b191dG0obGF0aXR1ZGU6IGZsb2F0LCBsb25naXR1ZGU6IGZsb2F0KSAtPiBUdXBsZVtmbG9hdCwgZmxvYXQsIGludCwgc3RyXToKICAgICIiIgogICAgQ29udmVydCBXR1M4NCBsYXQvbG9uIHRvIFVUTSBjb29yZGluYXRlcyAobWV0ZXJzKS4KCiAgICBSZXR1cm5zOgogICAgICAgIChlYXN0aW5nLCBub3J0aGluZywgem9uZV9udW1iZXIsIGhlbWlzcGhlcmUpCgogICAgTm90ZTogUmVxdWlyZXMgcHlwcm9qLiBGb3IgbWV0cmljIGRpc3RhbmNlIGNhbGN1bGF0aW9ucyBuZWVkZWQgaW4gcm91dGluZy4KICAgICIiIgogICAgdHJ5OgogICAgICAgIGZyb20gcHlwcm9qIGltcG9ydCBUcmFuc2Zvcm1lcgogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHJhaXNlIEltcG9ydEVycm9yKCJweXByb2ogcGFrZXRpIGdlcmVrbGk6IHBpcCBpbnN0YWxsIHB5cHJvaiIpCgogICAgem9uZSwgaGVtaSA9IGdldF91dG1fem9uZShsb25naXR1ZGUsIGxhdGl0dWRlKQogICAgZXBzZyA9IDMyNjAwICsgem9uZSBpZiBoZW1pID09ICJOIiBlbHNlIDMyNzAwICsgem9uZQoKICAgIHRyYW5zZm9ybWVyID0gVHJhbnNmb3JtZXIuZnJvbV9jcnMoIkVQU0c6NDMyNiIsIGYiRVBTRzp7ZXBzZ30iLCBhbHdheXNfeHk9VHJ1ZSkKICAgIGVhc3RpbmcsIG5vcnRoaW5nID0gdHJhbnNmb3JtZXIudHJhbnNmb3JtKGxvbmdpdHVkZSwgbGF0aXR1ZGUpCiAgICByZXR1cm4gZWFzdGluZywgbm9ydGhpbmcsIHpvbmUsIGhlbWkKCgpkZWYgdXRtX3RvX3dnczg0KAogICAgZWFzdGluZzogZmxvYXQsCiAgICBub3J0aGluZzogZmxvYXQsCiAgICB6b25lX251bWJlcjogaW50LAogICAgaGVtaXNwaGVyZTogc3RyID0gIk4iLAopIC0+IFR1cGxlW2Zsb2F0LCBmbG9hdF06CiAgICAiIiIKICAgIENvbnZlcnQgVVRNIGNvb3JkaW5hdGVzIGJhY2sgdG8gV0dTODQgbGF0L2xvbi4KICAgICIiIgogICAgdHJ5OgogICAgICAgIGZyb20gcHlwcm9qIGltcG9ydCBUcmFuc2Zvcm1lcgogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHJhaXNlIEltcG9ydEVycm9yKCJweXByb2ogcGFrZXRpIGdlcmVrbGkiKQoKICAgIGVwc2cgPSAzMjYwMCArIHpvbmVfbnVtYmVyIGlmIGhlbWlzcGhlcmUgPT0gIk4iIGVsc2UgMzI3MDAgKyB6b25lX251bWJlcgogICAgdHJhbnNmb3JtZXIgPSBUcmFuc2Zvcm1lci5mcm9tX2NycyhmIkVQU0c6e2Vwc2d9IiwgIkVQU0c6NDMyNiIsIGFsd2F5c194eT1UcnVlKQogICAgbG9uZ2l0dWRlLCBsYXRpdHVkZSA9IHRyYW5zZm9ybWVyLnRyYW5zZm9ybShlYXN0aW5nLCBub3J0aGluZykKICAgIHJldHVybiBsYXRpdHVkZSwgbG9uZ2l0dWRlCgoKZGVmIGhhdmVyc2luZV9kaXN0YW5jZSgKICAgIGxhdDE6IGZsb2F0LCBsb24xOiBmbG9hdCwKICAgIGxhdDI6IGZsb2F0LCBsb24yOiBmbG9hdCwKKSAtPiBmbG9hdDoKICAgICIiIgogICAgR3JlYXQtY2lyY2xlIGRpc3RhbmNlIGJldHdlZW4gdHdvIFdHUzg0IHBvaW50cyBpbiBNRVRFUlMuCiAgICBVc2VzIHRoZSBIYXZlcnNpbmUgZm9ybXVsYS4KICAgICIiIgogICAgUiA9IDYzNzEwMDAuMCAgIyBFYXJ0aCByYWRpdXMgaW4gbWV0ZXJzCiAgICBwaGkxID0gbWF0aC5yYWRpYW5zKGxhdDEpCiAgICBwaGkyID0gbWF0aC5yYWRpYW5zKGxhdDIpCiAgICBkcGhpID0gbWF0aC5yYWRpYW5zKGxhdDIgLSBsYXQxKQogICAgZGxhbSA9IG1hdGgucmFkaWFucyhsb24yIC0gbG9uMSkKICAgIGEgPSBtYXRoLnNpbihkcGhpIC8gMikgKiogMiArIG1hdGguY29zKHBoaTEpICogbWF0aC5jb3MocGhpMikgKiBtYXRoLnNpbihkbGFtIC8gMikgKiogMgogICAgYyA9IDIgKiBtYXRoLmF0YW4yKG1hdGguc3FydChhKSwgbWF0aC5zcXJ0KDEgLSBhKSkKICAgIHJldHVybiBSICogYwoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgSU1BR0UgTU9TQUlDS0lORwojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKZGVmIGVzdGltYXRlX2Ryb25lX2Zvb3RwcmludCgKICAgIGFsdGl0dWRlX206IGZsb2F0LAogICAgZm9jYWxfbGVuZ3RoX21tOiBmbG9hdCwKICAgIHNlbnNvcl93aWR0aF9tbTogZmxvYXQsCiAgICBpbWFnZV93aWR0aF9weDogaW50LAogICAgaW1hZ2VfaGVpZ2h0X3B4OiBpbnQsCikgLT4gVHVwbGVbZmxvYXQsIGZsb2F0LCBmbG9hdF06CiAgICAiIiIKICAgIEVzdGltYXRlIHRoZSBncm91bmQgZm9vdHByaW50IG9mIGEgZHJvbmUgaW1hZ2UuCgogICAgQXJnczoKICAgICAgICBhbHRpdHVkZV9tOiBkcm9uZSBhbHRpdHVkZSBhYm92ZSBncm91bmQgaW4gbWV0ZXJzCiAgICAgICAgZm9jYWxfbGVuZ3RoX21tOiBjYW1lcmEgZm9jYWwgbGVuZ3RoCiAgICAgICAgc2Vuc29yX3dpZHRoX21tOiBjYW1lcmEgc2Vuc29yIHdpZHRoIChlLmcuLCBESkkgUGhhbnRvbSA0OiAxMy4yIG1tKQogICAgICAgIGltYWdlX3dpZHRoX3B4OiBpbWFnZSB3aWR0aCBpbiBwaXhlbHMKICAgICAgICBpbWFnZV9oZWlnaHRfcHg6IGltYWdlIGhlaWdodCBpbiBwaXhlbHMKCiAgICBSZXR1cm5zOgogICAgICAgIChmb290cHJpbnRfd2lkdGhfbSwgZm9vdHByaW50X2hlaWdodF9tLCBnc2RfY21fcGVyX3BpeGVsKQogICAgICAgIEdTRCA9IEdyb3VuZCBTYW1wbGUgRGlzdGFuY2UKICAgICIiIgogICAgIyBHcm91bmQgc2FtcGxlIGRpc3RhbmNlOiBob3cgbWFueSBjbSBlYWNoIHBpeGVsIHJlcHJlc2VudHMKICAgIGdzZF9jbSA9IChzZW5zb3Jfd2lkdGhfbW0gKiBhbHRpdHVkZV9tICogMTAwKSAvIChmb2NhbF9sZW5ndGhfbW0gKiBpbWFnZV93aWR0aF9weCkKCiAgICAjIFRvdGFsIGZvb3RwcmludAogICAgZm9vdHByaW50X3dpZHRoX20gPSAoZ3NkX2NtICogaW1hZ2Vfd2lkdGhfcHgpIC8gMTAwCiAgICBmb290cHJpbnRfaGVpZ2h0X20gPSAoZ3NkX2NtICogaW1hZ2VfaGVpZ2h0X3B4KSAvIDEwMAoKICAgIHJldHVybiBmb290cHJpbnRfd2lkdGhfbSwgZm9vdHByaW50X2hlaWdodF9tLCBnc2RfY20KCgpkZWYgYnVpbGRfaW1hZ2VfaW5kZXgoaW1hZ2VfZGlyOiBzdHIsIGV4dGVuc2lvbnM6IFR1cGxlW3N0ciwgLi4uXSA9ICgiLmpwZyIsICIuanBlZyIsICIucG5nIiwgIi50aWYiLCAiLnRpZmYiKSkgLT4gTGlzdFtEaWN0XToKICAgICIiIgogICAgV2FsayBhIGRpcmVjdG9yeSBhbmQgYnVpbGQgYW4gaW5kZXggb2YgYWxsIGltYWdlcyB3aXRoIHRoZWlyIEdQUyBjb29yZGluYXRlcwogICAgKGlmIGF2YWlsYWJsZSBmcm9tIEVYSUYgb3IgR2VvVElGRiBtZXRhZGF0YSkuCgogICAgUmV0dXJuczoKICAgICAgICBMaXN0IG9mIGRpY3RzIHdpdGgga2V5czogcGF0aCwgZmlsZW5hbWUsIGhhc19ncHMsIGxhdGl0dWRlLCBsb25naXR1ZGUsCiAgICAgICAgYWx0aXR1ZGUsIHNvdXJjZSAoJ2V4aWYnIG9yICdnZW90aWZmJykKICAgICIiIgogICAgaW1hZ2VfZGlyID0gUGF0aChpbWFnZV9kaXIpCiAgICBpZiBub3QgaW1hZ2VfZGlyLmV4aXN0cygpOgogICAgICAgIHJldHVybiBbXQoKICAgIHJlY29yZHMgPSBbXQogICAgZm9yIGV4dCBpbiBleHRlbnNpb25zOgogICAgICAgIGZvciBpbWdfcGF0aCBpbiBpbWFnZV9kaXIucmdsb2IoZiIqe2V4dH0iKToKICAgICAgICAgICAgcmVjb3JkID0gewogICAgICAgICAgICAgICAgInBhdGgiOiBzdHIoaW1nX3BhdGgpLAogICAgICAgICAgICAgICAgImZpbGVuYW1lIjogaW1nX3BhdGgubmFtZSwKICAgICAgICAgICAgICAgICJoYXNfZ3BzIjogRmFsc2UsCiAgICAgICAgICAgICAgICAibGF0aXR1ZGUiOiBOb25lLAogICAgICAgICAgICAgICAgImxvbmdpdHVkZSI6IE5vbmUsCiAgICAgICAgICAgICAgICAiYWx0aXR1ZGUiOiBOb25lLAogICAgICAgICAgICAgICAgInNvdXJjZSI6IE5vbmUsCiAgICAgICAgICAgIH0KCiAgICAgICAgICAgICMgVHJ5IEVYSUYgZmlyc3QgKGRyb25lIEpQRUdzKQogICAgICAgICAgICBpZiBleHQubG93ZXIoKSBpbiAoIi5qcGciLCAiLmpwZWciKToKICAgICAgICAgICAgICAgIGdwcyA9IHJlYWRfZXhpZl9ncHMoc3RyKGltZ19wYXRoKSkKICAgICAgICAgICAgICAgIGlmIGdwcyBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgICAgICByZWNvcmRbImhhc19ncHMiXSA9IFRydWUKICAgICAgICAgICAgICAgICAgICByZWNvcmRbImxhdGl0dWRlIl0gPSBncHNbImxhdGl0dWRlIl0KICAgICAgICAgICAgICAgICAgICByZWNvcmRbImxvbmdpdHVkZSJdID0gZ3BzWyJsb25naXR1ZGUiXQogICAgICAgICAgICAgICAgICAgIHJlY29yZFsiYWx0aXR1ZGUiXSA9IGdwcy5nZXQoImFsdGl0dWRlIikKICAgICAgICAgICAgICAgICAgICByZWNvcmRbInNvdXJjZSJdID0gImV4aWYiCgogICAgICAgICAgICAjIFRyeSBHZW9USUZGCiAgICAgICAgICAgIGVsaWYgZXh0Lmxvd2VyKCkgaW4gKCIudGlmIiwgIi50aWZmIik6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgbWV0YSA9IHJlYWRfZ2VvdGlmZl9tZXRhZGF0YShzdHIoaW1nX3BhdGgpKQogICAgICAgICAgICAgICAgICAgIGlmIG1ldGEuZ2V0KCJjcnMiKToKICAgICAgICAgICAgICAgICAgICAgICAgIyBDZW50ZXIgb2YgYm91bmRzCiAgICAgICAgICAgICAgICAgICAgICAgIGIgPSBtZXRhWyJib3VuZHMiXQogICAgICAgICAgICAgICAgICAgICAgICBjeCA9IChiWyJsZWZ0Il0gKyBiWyJyaWdodCJdKSAvIDIKICAgICAgICAgICAgICAgICAgICAgICAgY3kgPSAoYlsiYm90dG9tIl0gKyBiWyJ0b3AiXSkgLyAyCgogICAgICAgICAgICAgICAgICAgICAgICAjIElmIENSUyBpcyBub3QgV0dTODQsIGNvbnZlcnQKICAgICAgICAgICAgICAgICAgICAgICAgaWYgIjQzMjYiIG5vdCBpbiBzdHIobWV0YVsiY3JzIl0pOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZyb20gcHlwcm9qIGltcG9ydCBUcmFuc2Zvcm1lcgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHQgPSBUcmFuc2Zvcm1lci5mcm9tX2NycyhtZXRhWyJjcnMiXSwgIkVQU0c6NDMyNiIsIGFsd2F5c194eT1UcnVlKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGxvbiwgbGF0ID0gdC50cmFuc2Zvcm0oY3gsIGN5KQogICAgICAgICAgICAgICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBsYXQsIGxvbiA9IGN5LCBjeAogICAgICAgICAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgbGF0LCBsb24gPSBjeSwgY3gKCiAgICAgICAgICAgICAgICAgICAgICAgIHJlY29yZFsiaGFzX2dwcyJdID0gVHJ1ZQogICAgICAgICAgICAgICAgICAgICAgICByZWNvcmRbImxhdGl0dWRlIl0gPSBsYXQKICAgICAgICAgICAgICAgICAgICAgICAgcmVjb3JkWyJsb25naXR1ZGUiXSA9IGxvbgogICAgICAgICAgICAgICAgICAgICAgICByZWNvcmRbInNvdXJjZSJdID0gImdlb3RpZmYiCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgICAgIHBhc3MKCiAgICAgICAgICAgIHJlY29yZHMuYXBwZW5kKHJlY29yZCkKCiAgICByZXR1cm4gcmVjb3Jkcwo=").decode("utf-8")

geo_path = os.path.join(SRC_DIR, "geo_utils.py")
with open(geo_path, "w") as f:
    f.write(GEO_UTILS_PY)

size_kb = os.path.getsize(geo_path) / 1024
print(f"✅ geo_utils.py yazıldı ({size_kb:.1f} KB)")
print(f"   Yol: {geo_path}")

# Python yoluna ekle (zaten ekli olabilir, idempotent)
import sys
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Import testi
print("\n🧪 Import testi...")
import importlib
import geo_utils
importlib.reload(geo_utils)

from geo_utils import (
    read_exif_gps,
    read_exif_camera_info,
    read_geotiff_metadata,
    read_geotiff_array,
    pixel_to_geo,
    geo_to_pixel,
    pixel_polygon_to_geo,
    wgs84_to_utm,
    utm_to_wgs84,
    haversine_distance,
    estimate_drone_footprint,
    build_image_index,
    get_utm_zone,
)
print("  ✅ Tüm fonksiyonlar import edildi")

✅ geo_utils.py yazıldı (13.9 KB)
   Yol: /content/drive/MyDrive/AFETSONAR/src/geo_utils.py

🧪 Import testi...
  ✅ Tüm fonksiyonlar import edildi


## 4️⃣ Koordinat Dönüşümü Testleri

### Bu hücre ne yapıyor?

`geo_utils` modülünün matematiğinin **doğru çalıştığını** doğruluyor. 4 test yapıyoruz:

### Test 1: WGS84 → UTM → WGS84 (round-trip)

Bir koordinatı UTM'e çevirip geri çevirdiğimizde aynı değeri almalıyız (küçük sayısal hata kabul edilebilir).

### Test 2: Haversine mesafe

İstanbul (Sultanahmet) ve Ankara (Anıtkabir) arası gerçek mesafe ~351 km. Fonksiyonumuz bunu doğru hesaplıyor mu?

### Test 3: UTM zone tespiti

İstanbul (lon=28.97) → UTM zone 35N olmalı. Tokyo (lon=139.69) → 54N. New York (lon=-74.0) → 18N.

### Test 4: Pixel ↔ Geo dönüşümü (sentetik)

Bir affine transform tanımla, pixel→geo→pixel round trip yap, kayıp olmadığını doğrula.

In [4]:
import importlib
import geo_utils
importlib.reload(geo_utils)
from geo_utils import (
    wgs84_to_utm, utm_to_wgs84, haversine_distance,
    get_utm_zone, pixel_to_geo, geo_to_pixel
)

print("🧪 Test 1: WGS84 → UTM → WGS84 round-trip")
print("=" * 50)

test_points = [
    ("İstanbul (Sultanahmet)", 41.0086, 28.9802),
    ("Ankara (Anıtkabir)",     39.9255, 32.8378),
    ("İzmir (Konak)",          38.4192, 27.1287),
    ("Tokyo (Imperial)",       35.6852, 139.7528),
    ("New York (Empire)",     40.7484, -73.9857),
]

for name, lat, lon in test_points:
    e, n, zone, hemi = wgs84_to_utm(lat, lon)
    lat_back, lon_back = utm_to_wgs84(e, n, zone, hemi)
    err_m = haversine_distance(lat, lon, lat_back, lon_back)
    print(f"  {name:25s} zone {zone}{hemi}, round-trip error: {err_m*1000:.3f} mm")
    assert err_m < 0.001, f"Round-trip error too large: {err_m} m"

print("✅ Tüm round-trip testleri başarılı (< 1 mm hata)")

print("\n\n🧪 Test 2: Haversine mesafe")
print("=" * 50)

# İstanbul - Ankara
ist_lat, ist_lon = 41.0086, 28.9802
ank_lat, ank_lon = 39.9255, 32.8378
dist_m = haversine_distance(ist_lat, ist_lon, ank_lat, ank_lon)
dist_km = dist_m / 1000
print(f"  İstanbul - Ankara: {dist_km:.1f} km (gerçek ~351 km)")
assert 340 < dist_km < 360, f"Mesafe yanlış: {dist_km}"

# İstanbul - İzmir
izm_lat, izm_lon = 38.4192, 27.1287
dist_m2 = haversine_distance(ist_lat, ist_lon, izm_lat, izm_lon)
print(f"  İstanbul - İzmir:  {dist_m2/1000:.1f} km (gerçek ~324 km)")

print("✅ Haversine mesafe doğru")

print("\n\n🧪 Test 3: UTM zone tespiti")
print("=" * 50)

zones = [
    (41.0086, 28.9802, 35, "N"),  # İstanbul
    (39.9255, 32.8378, 36, "N"),  # Ankara
    (35.6852, 139.7528, 54, "N"), # Tokyo
    (40.7484, -73.9857, 18, "N"), # NYC
    (-33.8688, 151.2093, 56, "S"),# Sydney
]

for lat, lon, expected_zone, expected_hemi in zones:
    zone, hemi = get_utm_zone(lon, lat)
    marker = "✅" if (zone == expected_zone and hemi == expected_hemi) else "❌"
    print(f"  {marker} lat={lat:7.2f}, lon={lon:8.2f} → zone {zone}{hemi}  (beklenen {expected_zone}{expected_hemi})")
    assert zone == expected_zone and hemi == expected_hemi

print("✅ Tüm zone tespitleri doğru")

print("\n\n🧪 Test 4: Pixel ↔ Geo dönüşümü")
print("=" * 50)

# Sentetik bir GeoTIFF affine transform:
# Sol üst köşe lat=41.0, lon=29.0
# Pixel boyutu: 10 metre (yaklaşık 0.0001 derece enlem, 0.00013 derece boylam İstanbul'da)
# Format: [pixel_width, 0, x_origin, 0, -pixel_height, y_origin]
synthetic_transform = [0.0001, 0.0, 29.0, 0.0, -0.0001, 41.0]

test_pixels = [(0, 0), (100, 100), (500, 250), (1024, 768)]
print(f"  Sentetik transform: pixel boyutu 0.0001° (~10 m İstanbul'da)")
print()

for px, py in test_pixels:
    x, y = pixel_to_geo(px, py, synthetic_transform)
    px_back, py_back = geo_to_pixel(x, y, synthetic_transform)
    print(f"  pixel ({px:4d},{py:4d}) → geo ({x:.6f},{y:.6f}) → pixel ({px_back:.2f},{py_back:.2f})")
    assert abs(px - px_back) < 1e-6
    assert abs(py - py_back) < 1e-6

print("\n✅ Pixel ↔ Geo dönüşümü mükemmel (< 1e-6 hata)")

🧪 Test 1: WGS84 → UTM → WGS84 round-trip
  İstanbul (Sultanahmet)    zone 35N, round-trip error: 0.000 mm
  Ankara (Anıtkabir)        zone 36N, round-trip error: 0.000 mm
  İzmir (Konak)             zone 35N, round-trip error: 0.000 mm
  Tokyo (Imperial)          zone 54N, round-trip error: 0.000 mm
  New York (Empire)         zone 18N, round-trip error: 0.000 mm
✅ Tüm round-trip testleri başarılı (< 1 mm hata)


🧪 Test 2: Haversine mesafe
  İstanbul - Ankara: 347.8 km (gerçek ~351 km)
  İstanbul - İzmir:  328.6 km (gerçek ~324 km)
✅ Haversine mesafe doğru


🧪 Test 3: UTM zone tespiti
  ✅ lat=  41.01, lon=   28.98 → zone 35N  (beklenen 35N)
  ✅ lat=  39.93, lon=   32.84 → zone 36N  (beklenen 36N)
  ✅ lat=  35.69, lon=  139.75 → zone 54N  (beklenen 54N)
  ✅ lat=  40.75, lon=  -73.99 → zone 18N  (beklenen 18N)
  ✅ lat= -33.87, lon=  151.21 → zone 56S  (beklenen 56S)
✅ Tüm zone tespitleri doğru


🧪 Test 4: Pixel ↔ Geo dönüşümü
  Sentetik transform: pixel boyutu 0.0001° (~10 m İstanbul'da)

## 5️⃣ Sentetik GeoTIFF Üretme ve Okuma

### Bu hücre ne yapıyor?

Test için **kendi GeoTIFF dosyamızı üretiyoruz**. Sebep:

- Drone görüntün yoksa
- xBD görüntülerinin EXIF/coğrafi metadata'sı yok (sadece pixel)
- Yine de geo_utils'i test edebilmemiz lazım

Sentetik veri:
- 1024×1024 pixel renkli görüntü (rastgele desenli)
- WGS84 koordinat sisteminde
- İstanbul Sultanahmet çevresinde konumlandırılmış (~41.005°N, 28.975°E)
- 1 pixel = 0.5 metre (yaklaşık)

Bu dosyayı Drive'a yazıyoruz, sonra `geo_utils.read_geotiff_metadata` ile okuyoruz, ve doğru bilgileri çıkardığımızı doğruluyoruz.

### Beklenen çıktı

```
✅ Sentetik GeoTIFF üretildi: .../test_geo/sultanahmet_test.tif
   Boyut: 1024 x 1024
   CRS: EPSG:4326

📋 Metadata okundu:
   crs: EPSG:4326
   bounds: lat 41.003-41.008, lon 28.972-28.977
   pixel_size: ~0.5 m
   ...
```

In [5]:
import numpy as np
import rasterio
from rasterio.transform import from_origin

print("🎨 Sentetik GeoTIFF üretiliyor...")
print("=" * 50)

# === Parametreler ===
WIDTH = 1024
HEIGHT = 1024

# Sultanahmet, İstanbul yakınında bir yer
# Sol üst köşe (origin)
ORIGIN_LON = 28.9720
ORIGIN_LAT = 41.0080

# Pixel boyutu (yaklaşık 0.5 metre = 0.0000045° enlem, daha az boylam)
PIXEL_DEG = 0.0000045

# === Affine transform ===
# from_origin(west, north, xsize, ysize) — y artarken aşağı gider, ysize negatif değil
transform = from_origin(ORIGIN_LON, ORIGIN_LAT, PIXEL_DEG, PIXEL_DEG)

# === Rastgele bir RGB görüntü üret (binalar gibi gradyan + gürültü) ===
np.random.seed(42)
rgb = np.zeros((3, HEIGHT, WIDTH), dtype=np.uint8)

# Ana renk: yeşil-gri (yer)
rgb[0] = 80   # R
rgb[1] = 100  # G
rgb[2] = 70   # B

# Birkaç sahte "bina" çiz (kareler)
for _ in range(20):
    x = np.random.randint(50, WIDTH - 100)
    y = np.random.randint(50, HEIGHT - 100)
    w = np.random.randint(30, 80)
    h = np.random.randint(30, 80)
    rgb[0, y:y+h, x:x+w] = np.random.randint(120, 200)
    rgb[1, y:y+h, x:x+w] = np.random.randint(100, 180)
    rgb[2, y:y+h, x:x+w] = np.random.randint(80, 150)

# Gürültü ekle
noise = np.random.randint(-15, 15, rgb.shape, dtype=np.int16)
rgb = np.clip(rgb.astype(np.int16) + noise, 0, 255).astype(np.uint8)

# === GeoTIFF olarak kaydet ===
test_tiff_path = os.path.join(TEST_GEO_DIR, "sultanahmet_test.tif")

with rasterio.open(
    test_tiff_path,
    "w",
    driver="GTiff",
    height=HEIGHT,
    width=WIDTH,
    count=3,
    dtype=rgb.dtype,
    crs="EPSG:4326",  # WGS84
    transform=transform,
) as dst:
    dst.write(rgb)

print(f"✅ Sentetik GeoTIFF üretildi: {test_tiff_path}")
print(f"   Boyut: {WIDTH} × {HEIGHT}")
print(f"   CRS: EPSG:4326 (WGS84)")
print(f"   Origin: lat={ORIGIN_LAT}, lon={ORIGIN_LON}")
print(f"   Pixel boyutu: {PIXEL_DEG}° (~0.5 m)")

# === geo_utils ile geri oku ===
print("\n📋 geo_utils.read_geotiff_metadata ile okuma:")
print("=" * 50)

import importlib, geo_utils
importlib.reload(geo_utils)
from geo_utils import read_geotiff_metadata, read_geotiff_array, pixel_to_geo

meta = read_geotiff_metadata(test_tiff_path)
for key, val in meta.items():
    if isinstance(val, dict):
        print(f"  {key}:")
        for k, v in val.items():
            print(f"    {k}: {v}")
    else:
        print(f"  {key}: {val}")

# === Pixel → Geo testi: merkez pixeli ===
print("\n🎯 Merkez pixel'in koordinatı:")
center_x, center_y = WIDTH // 2, HEIGHT // 2
geo_x, geo_y = pixel_to_geo(center_x, center_y, meta["transform"])
print(f"   Pixel ({center_x}, {center_y}) → Geo (lon={geo_x:.6f}, lat={geo_y:.6f})")

# Doğrulama: tam ortada olmalı
expected_lon = ORIGIN_LON + (WIDTH / 2) * PIXEL_DEG
expected_lat = ORIGIN_LAT - (HEIGHT / 2) * PIXEL_DEG  # y aşağı doğru
print(f"   Beklenen:                      (lon={expected_lon:.6f}, lat={expected_lat:.6f})")

assert abs(geo_x - expected_lon) < 1e-9
assert abs(geo_y - expected_lat) < 1e-9
print("   ✅ Doğru!")

# === Array okuma testi ===
print("\n📊 Array olarak okuma:")
arr, meta2 = read_geotiff_array(test_tiff_path)
print(f"   Array shape: {arr.shape}")
print(f"   dtype: {arr.dtype}")
print(f"   min/max: {arr.min()}/{arr.max()}")

🎨 Sentetik GeoTIFF üretiliyor...
✅ Sentetik GeoTIFF üretildi: /content/drive/MyDrive/AFETSONAR/data/test_geo/sultanahmet_test.tif
   Boyut: 1024 × 1024
   CRS: EPSG:4326 (WGS84)
   Origin: lat=41.008, lon=28.972
   Pixel boyutu: 4.5e-06° (~0.5 m)

📋 geo_utils.read_geotiff_metadata ile okuma:
  crs: EPSG:4326
  bounds:
    left: 28.972
    bottom: 41.003392000000005
    right: 28.976608000000002
    top: 41.008
  transform: [4.5e-06, 0.0, 28.972, 0.0, -4.5e-06, 41.008]
  width: 1024
  height: 1024
  count: 3
  dtype: uint8
  pixel_size_x: 4.5e-06
  pixel_size_y: 4.5e-06

🎯 Merkez pixel'in koordinatı:
   Pixel (512, 512) → Geo (lon=28.974304, lat=41.005696)
   Beklenen:                      (lon=28.974304, lat=41.005696)
   ✅ Doğru!

📊 Array olarak okuma:
   Array shape: (1024, 1024, 3)
   dtype: uint8
   min/max: 55/211


## 6️⃣ Drone Footprint Hesabı

### Bu hücre ne yapıyor?

Bir drone uçarken bir görüntü çekiyor. **Bu görüntü yerde ne kadar alan kapsıyor?**

Bunu bilmek lazım çünkü:
- A* rota planlamasında hasar bölgelerinin gerçek alanını bilmemiz lazım
- Birden fazla drone fotoğrafını birleştirirken (mozaik) örtüşmeleri hesaplamak için
- Modelin bina tespiti yaptığında, binanın **gerçek metrekaresini** raporlamak için

### Formül (GSD — Ground Sample Distance)

```
GSD (cm/pixel) = (sensor_width_mm × altitude_m × 100) / (focal_length_mm × image_width_px)
```

GSD bize "1 pixel kaç cm yere karşılık geliyor" der.

### Örnek hesap: DJI Phantom 4 Pro

| Parametre | Değer |
|---|---|
| Sensor width | 13.2 mm |
| Focal length | 8.8 mm |
| Image width | 5472 pixel |
| Yükseklik | 100 m |

→ GSD = (13.2 × 100 × 100) / (8.8 × 5472) ≈ **2.74 cm/pixel**
→ Footprint width = 2.74 × 5472 / 100 ≈ **150 metre**
→ Footprint height (3648 px) ≈ **100 metre**

Yani bir drone 100 metre yükseklikte uçarken bir fotoğrafta ~150×100 metre yer kaplıyor, ve her pixel ~3 cm.

In [6]:
import importlib, geo_utils
importlib.reload(geo_utils)
from geo_utils import estimate_drone_footprint

print("🚁 Drone footprint hesabı")
print("=" * 50)

# Yaygın drone modelleri
drones = [
    {
        "name": "DJI Phantom 4 Pro",
        "sensor_width_mm": 13.2,
        "focal_length_mm": 8.8,
        "image_width_px": 5472,
        "image_height_px": 3648,
    },
    {
        "name": "DJI Mavic 3",
        "sensor_width_mm": 17.3,
        "focal_length_mm": 12.29,
        "image_width_px": 5280,
        "image_height_px": 3956,
    },
    {
        "name": "DJI Mini 3 Pro",
        "sensor_width_mm": 9.6,
        "focal_length_mm": 6.7,
        "image_width_px": 4032,
        "image_height_px": 3024,
    },
    {
        "name": "Autel EVO Lite+",
        "sensor_width_mm": 13.2,
        "focal_length_mm": 10.0,
        "image_width_px": 5472,
        "image_height_px": 3648,
    },
]

altitudes = [50, 100, 150, 200]

for drone in drones:
    print(f"\n📷 {drone['name']}")
    print(f"   Sensor: {drone['sensor_width_mm']} mm | Focal: {drone['focal_length_mm']} mm | Image: {drone['image_width_px']}×{drone['image_height_px']}")
    print(f"   {'Yükseklik':<12} {'Footprint':<20} {'GSD (cm/px)':<12} {'Bina (50m²)':<15}")
    print(f"   {'-'*60}")

    for alt in altitudes:
        w, h, gsd = estimate_drone_footprint(
            altitude_m=alt,
            focal_length_mm=drone["focal_length_mm"],
            sensor_width_mm=drone["sensor_width_mm"],
            image_width_px=drone["image_width_px"],
            image_height_px=drone["image_height_px"],
        )
        # 50 m² lik bir bina kaç pixel'e karşılık gelir?
        building_pixels = (50 * 10000) / (gsd ** 2)
        print(f"   {alt} m{' '*7} {w:6.0f}×{h:.0f} m{' '*8} {gsd:6.2f}      {building_pixels:>8.0f} pixel")

print("\n\n💡 Pratik kural:")
print("   • 50-100 m yükseklik → bina detayları net (GSD < 5 cm)")
print("   • 150-200 m yükseklik → genel hasar tarama (daha geniş alan)")
print("   • Çatı hasarı için GSD < 5 cm önerilir")

🚁 Drone footprint hesabı

📷 DJI Phantom 4 Pro
   Sensor: 13.2 mm | Focal: 8.8 mm | Image: 5472×3648
   Yükseklik    Footprint            GSD (cm/px)  Bina (50m²)    
   ------------------------------------------------------------
   50 m            75×50 m           1.37        266158 pixel
   100 m           150×100 m           2.74         66540 pixel
   150 m           225×150 m           4.11         29573 pixel
   200 m           300×200 m           5.48         16635 pixel

📷 DJI Mavic 3
   Sensor: 17.3 mm | Focal: 12.29 mm | Image: 5280×3956
   Yükseklik    Footprint            GSD (cm/px)  Bina (50m²)    
   ------------------------------------------------------------
   50 m            70×53 m           1.33        281390 pixel
   100 m           141×105 m           2.67         70348 pixel
   150 m           211×158 m           4.00         31266 pixel
   200 m           282×211 m           5.33         17587 pixel

📷 DJI Mini 3 Pro
   Sensor: 9.6 mm | Focal: 6.7 mm | Image: 

## 7️⃣ Drone EXIF GPS Test

### Bu hücre ne yapıyor?

Drone JPEG'i **simüle ediyor**. Gerçek bir DJI drone JPEG'i elimizde yok ama:

1. Bir görüntüyü oluştur
2. PIL ile EXIF GPS metadata'sını **manuel** ekle (bir drone gibi)
3. `geo_utils.read_exif_gps()` ile okumayı dene
4. Çıkan koordinatın yazdığımızla eşleştiğini doğrula

### Pratik

Sen **gerçek bir drone fotoğrafına sahipsen** (kendi drone'un veya internetten bir DJI sample), test hücresinin altında onun yolunu verip okuyabilirsin. Hücrenin sonunda buna örnek var.

### EXIF GPS formatı

EXIF, GPS'i şöyle saklar:
- Latitude: `(40, 12, 30.5)` → 40° 12' 30.5" (derece-dakika-saniye)
- LatitudeRef: `'N'` veya `'S'`
- Longitude: aynı format
- LongitudeRef: `'E'` veya `'W'`

`geo_utils.read_exif_gps` bunu otomatik decimal degree'ye çevirir.

In [7]:
print("🧪 EXIF GPS Test (sentetik drone fotoğrafı)")
print("=" * 50)

# === Sentetik bir JPEG + EXIF GPS oluştur ===
from PIL import Image

# piexif Colab'da olmayabilir
try:
    import piexif
except ImportError:
    print("📥 piexif kuruluyor...")
    !pip install -q piexif
    import piexif

# Hedef koordinat: Beyazıt Meydanı, İstanbul
TARGET_LAT = 41.0108
TARGET_LON = 28.9650
TARGET_ALT = 50.0  # 50 metre yükseklik

def deg_to_dms_rational(deg_float):
    # Decimal degrees -> EXIF rational DMS format
    deg_abs = abs(deg_float)
    d = int(deg_abs)
    m_float = (deg_abs - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    # EXIF rational: (numerator, denominator)
    return ((d, 1), (m, 1), (int(s * 10000), 10000))

# Görüntü
img = Image.new("RGB", (640, 480), color=(100, 150, 100))

# EXIF dict
gps_ifd = {
    piexif.GPSIFD.GPSLatitudeRef: "N" if TARGET_LAT >= 0 else "S",
    piexif.GPSIFD.GPSLatitude: deg_to_dms_rational(TARGET_LAT),
    piexif.GPSIFD.GPSLongitudeRef: "E" if TARGET_LON >= 0 else "W",
    piexif.GPSIFD.GPSLongitude: deg_to_dms_rational(TARGET_LON),
    piexif.GPSIFD.GPSAltitudeRef: 0,  # 0 = above sea level
    piexif.GPSIFD.GPSAltitude: (int(TARGET_ALT * 100), 100),
}

zeroth_ifd = {
    piexif.ImageIFD.Make: "DJI",
    piexif.ImageIFD.Model: "FC6310 (Phantom 4 Pro)",
}

exif_dict = {"0th": zeroth_ifd, "Exif": {}, "GPS": gps_ifd, "1st": {}, "thumbnail": None}
exif_bytes = piexif.dump(exif_dict)

# Kaydet
test_jpg_path = os.path.join(TEST_GEO_DIR, "synthetic_drone.jpg")
img.save(test_jpg_path, "jpeg", exif=exif_bytes)

print(f"✅ Sentetik drone JPEG oluşturuldu: {test_jpg_path}")
print(f"   Hedef GPS: lat={TARGET_LAT}, lon={TARGET_LON}, alt={TARGET_ALT} m")

# === geo_utils ile oku ===
import importlib, geo_utils
importlib.reload(geo_utils)
from geo_utils import read_exif_gps, read_exif_camera_info

print("\n📡 geo_utils.read_exif_gps ile okuma:")
gps = read_exif_gps(test_jpg_path)
if gps is None:
    print("   ❌ GPS okunamadı")
else:
    print(f"   Latitude:  {gps['latitude']:.6f}  (beklenen {TARGET_LAT})")
    print(f"   Longitude: {gps['longitude']:.6f}  (beklenen {TARGET_LON})")
    print(f"   Altitude:  {gps['altitude']:.2f} m  (beklenen {TARGET_ALT})")

    err_lat = abs(gps['latitude'] - TARGET_LAT)
    err_lon = abs(gps['longitude'] - TARGET_LON)
    print(f"\n   Latitude error:  {err_lat:.8f}°")
    print(f"   Longitude error: {err_lon:.8f}°")

    if err_lat < 1e-5 and err_lon < 1e-5:
        print("   ✅ GPS okuma doğru!")
    else:
        print("   ⚠️ GPS hatası beklenenden büyük")

# === Camera info ===
print("\n📷 Kamera bilgisi:")
info = read_exif_camera_info(test_jpg_path)
for k, v in info.items():
    print(f"   {k}: {v}")

print("\n\n💡 Gerçek drone fotoğrafı testi (opsiyonel)")
print("=" * 50)
print("Eğer kendi drone fotoğrafın varsa şöyle test edebilirsin:")
print("")
print("  # Drive'a fotoğrafını yükle (örn. AFETSONAR/data/test_geo/my_drone.jpg)")
print("  my_drone_path = os.path.join(TEST_GEO_DIR, 'my_drone.jpg')")
print("  if os.path.exists(my_drone_path):")
print("      gps = read_exif_gps(my_drone_path)")
print("      info = read_exif_camera_info(my_drone_path)")
print("      print('GPS:', gps)")
print("      print('Kamera:', info)")

🧪 EXIF GPS Test (sentetik drone fotoğrafı)
📥 piexif kuruluyor...
✅ Sentetik drone JPEG oluşturuldu: /content/drive/MyDrive/AFETSONAR/data/test_geo/synthetic_drone.jpg
   Hedef GPS: lat=41.0108, lon=28.965, alt=50.0 m

📡 geo_utils.read_exif_gps ile okuma:
   Latitude:  41.010800  (beklenen 41.0108)
   Longitude: 28.965000  (beklenen 28.965)
   Altitude:  50.00 m  (beklenen 50.0)

   Latitude error:  0.00000000°
   Longitude error: 0.00000003°
   ✅ GPS okuma doğru!

📷 Kamera bilgisi:
   make: DJI
   model: FC6310 (Phantom 4 Pro)


💡 Gerçek drone fotoğrafı testi (opsiyonel)
Eğer kendi drone fotoğrafın varsa şöyle test edebilirsin:

  # Drive'a fotoğrafını yükle (örn. AFETSONAR/data/test_geo/my_drone.jpg)
  my_drone_path = os.path.join(TEST_GEO_DIR, 'my_drone.jpg')
  if os.path.exists(my_drone_path):
      gps = read_exif_gps(my_drone_path)
      info = read_exif_camera_info(my_drone_path)
      print('GPS:', gps)
      print('Kamera:', info)


## 8️⃣ Bina Poligonu → Geo Koordinatlara Dönüştürme

### Bu hücre ne yapıyor?

Bu, **Notebook 5-7 için kritik** bir test. Senaryomuz:

1. Model bir uydu görüntüsü üzerinde tahmin yapıyor
2. Çıktı: pixel koordinatlarda binaları tespit ediyor (örn. her destroyed bina için poligon koordinatları)
3. Biz bu poligonları **harita üzerinde** göstermek istiyoruz (Folium)
4. Bunun için **pixel poligonlarını WGS84 lat/lon poligonlarına** çevirmemiz lazım

### Test akışı

1. Sentetik GeoTIFF'imizdeki transform'u kullan
2. 3 sahte bina poligonu tanımla (pixel koordinatlarda)
3. Her biri için pixel→geo dönüşümü yap
4. Geo poligonun gerçek metrekaresini hesapla
5. Folium harita oluştur, üzerinde göster

### Beklenen çıktı

3 kırmızı poligon (binalar) İstanbul üzerinde Folium haritasında gösterilen bir HTML.

In [8]:
import importlib, geo_utils
importlib.reload(geo_utils)
from geo_utils import (
    read_geotiff_metadata, pixel_polygon_to_geo,
    wgs84_to_utm, haversine_distance
)

# Sentetik GeoTIFF'in metadata'sını al
test_tiff_path = os.path.join(TEST_GEO_DIR, "sultanahmet_test.tif")
meta = read_geotiff_metadata(test_tiff_path)
transform = meta["transform"]

# === Sahte bina poligonları (pixel koordinatlarda) ===
# Modelimizin tahmini olarak düşün
fake_buildings = [
    {
        "id": 1,
        "damage": "destroyed",
        "pixel_polygon": [(150, 200), (250, 200), (250, 280), (150, 280)],  # 100x80 pixel
    },
    {
        "id": 2,
        "damage": "major-damage",
        "pixel_polygon": [(400, 350), (520, 350), (530, 460), (390, 460)],
    },
    {
        "id": 3,
        "damage": "minor-damage",
        "pixel_polygon": [(700, 600), (800, 600), (800, 700), (700, 700)],
    },
]

print("🏢 Bina poligonları işleniyor...")
print("=" * 50)

# Pixel → Geo dönüşümü
from shapely.geometry import Polygon as ShapelyPolygon

for bld in fake_buildings:
    geo_poly = pixel_polygon_to_geo(bld["pixel_polygon"], transform)
    bld["geo_polygon"] = geo_poly  # liste of (lon, lat)

    # Centroid
    lons = [p[0] for p in geo_poly]
    lats = [p[1] for p in geo_poly]
    center_lon = sum(lons) / len(lons)
    center_lat = sum(lats) / len(lats)
    bld["center"] = (center_lat, center_lon)

    # Gerçek alan (UTM'e çevirip metre cinsinden hesapla)
    utm_coords = []
    for lon, lat in geo_poly:
        e, n, _, _ = wgs84_to_utm(lat, lon)
        utm_coords.append((e, n))
    utm_poly = ShapelyPolygon(utm_coords)
    area_m2 = utm_poly.area

    bld["area_m2"] = area_m2

    print(f"\n  Bina {bld['id']} ({bld['damage']})")
    print(f"    Pixel polygon:  {bld['pixel_polygon']}")
    print(f"    Geo center:     lat={center_lat:.6f}, lon={center_lon:.6f}")
    print(f"    Area:           {area_m2:.1f} m²")

# === Folium ile haritada göster ===
print("\n\n🗺️  Folium haritası oluşturuluyor...")
print("=" * 50)

import folium

# Haritayı sentetik GeoTIFF'in merkezine konumlandır
center_lat = (meta["bounds"]["bottom"] + meta["bounds"]["top"]) / 2
center_lon = (meta["bounds"]["left"] + meta["bounds"]["right"]) / 2

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=19,
    tiles="OpenStreetMap",
)

# Damage rengi → folium rengi
damage_colors = {
    "destroyed": "red",
    "major-damage": "orange",
    "minor-damage": "yellow",
    "no-damage": "green",
}

for bld in fake_buildings:
    # Folium [lat, lon] bekler, geo_poly [lon, lat] verir → çevir
    folium_coords = [(lat, lon) for lon, lat in bld["geo_polygon"]]

    folium.Polygon(
        locations=folium_coords,
        color=damage_colors.get(bld["damage"], "gray"),
        weight=2,
        fill=True,
        fill_color=damage_colors.get(bld["damage"], "gray"),
        fill_opacity=0.5,
        popup=folium.Popup(
            f"<b>Bina #{bld['id']}</b><br>"
            f"Hasar: {bld['damage']}<br>"
            f"Alan: {bld['area_m2']:.1f} m²",
            max_width=200,
        ),
    ).add_to(m)

# Sentetik GeoTIFF sınırlarını da göster
folium.Rectangle(
    bounds=[
        [meta["bounds"]["bottom"], meta["bounds"]["left"]],
        [meta["bounds"]["top"], meta["bounds"]["right"]],
    ],
    color="blue",
    weight=2,
    fill=False,
    popup="Test GeoTIFF sınırları",
).add_to(m)

# Drive'a kaydet
map_path = os.path.join(OUTPUTS_VIZ, "test_buildings_map.html")
m.save(map_path)
print(f"✅ Folium harita kaydedildi: {map_path}")

# Notebook'ta göster
print("\nNotebook'ta gösterim:")
m

🏢 Bina poligonları işleniyor...

  Bina 1 (destroyed)
    Pixel polygon:  [(150, 200), (250, 200), (250, 280), (150, 280)]
    Geo center:     lat=41.006920, lon=28.972900
    Area:           1513.3 m²

  Bina 2 (major-damage)
    Pixel polygon:  [(400, 350), (520, 350), (530, 460), (390, 460)]
    Geo center:     lat=41.006177, lon=28.974070
    Area:           2705.1 m²

  Bina 3 (minor-damage)
    Pixel polygon:  [(700, 600), (800, 600), (800, 700), (700, 700)]
    Geo center:     lat=41.005075, lon=28.975375
    Area:           1891.7 m²


🗺️  Folium haritası oluşturuluyor...
✅ Folium harita kaydedildi: /content/drive/MyDrive/AFETSONAR/outputs/visualizations/test_buildings_map.html

Notebook'ta gösterim:


## 🎉 Notebook 4 Tamamlandı!

### Ne yaptık?

✅ `src/geo_utils.py` modülü Drive'a yazıldı (~14 KB)
✅ Coğrafi kütüphaneler doğrulandı (exifread, rasterio, pyproj, shapely)
✅ WGS84 ↔ UTM round-trip testleri (< 1 mm hata)
✅ Haversine mesafe doğrulandı (İstanbul-Ankara ~351 km)
✅ UTM zone tespiti 5 farklı şehir için doğru
✅ Pixel ↔ Geo dönüşümü matematiği test edildi
✅ Sentetik GeoTIFF üretildi (Sultanahmet, 1024×1024)
✅ Drone footprint formülü 4 model × 4 yükseklik tablosu
✅ Sentetik drone JPEG + EXIF GPS okuma
✅ Bina poligonları geo koordinatlara çevrildi, Folium haritasında gösterildi

### Drive'da şu an ne var?

```
AFETSONAR/
├── src/
│   └── geo_utils.py                    ✅ YENİ
├── data/test_geo/
│   ├── sultanahmet_test.tif            ✅ Sentetik GeoTIFF
│   └── synthetic_drone.jpg             ✅ Sentetik drone JPEG
└── outputs/visualizations/
    └── test_buildings_map.html         ✅ Test haritası
```

### Sonraki Adım

Şu an iki seçeneğimiz var:

**Seçenek A:** Notebook 5'e geç (Priority + Voronoi + FEMA debris)
- Hasar skorları, ekip atama, S(t)=S₀·exp(-λt) formülü
- Modelden bağımsız, hemen yazılabilir
- Eğitim devam ederken paralel hazırlık

**Seçenek B:** Öğretmen eğitiminin sonuçlarını bekle, sonra Notebook 3 (öğrenci + KD)
- Öğretmen mIoU sonucuna göre KD stratejisini ayarlayabiliriz

İkisini de yapacağız, sıralama sana kalmış. Bana söyle.

---

**Calamitas AI · Teknofest 2025 · Notebook 4/8** 🚀